In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.use_warehouse("NOVAMART_WH")
session.use_database("NOVAMART_DB")
session.use_schema("STAGING")

session.sql("""
    SELECT
        CURRENT_DATABASE() AS DATABASE_NAME,
        CURRENT_SCHEMA() AS SCHEMA_NAME,
        CURRENT_WAREHOUSE() AS WAREHOUSE_NAME
""").show()

In [ ]:
tables_and_keys = {
    "CUSTOMERS_RAW": ["CUSTOMER_ID"],
    "STORES_RAW": ["STORE_ID"],
    "PRODUCTS_RAW": ["PRODUCT_ID"],
    "ORDERS_RAW": ["ORDER_ID"],
    "ORDER_DETAILS_RAW": ["ORDER_DETAIL_ID"],
    "PAYMENTS_RAW": ["PAYMENT_ID"],
    "RETURNS_RAW": ["RETURN_ID"],
    "INVENTORY_RAW": ["STORE_ID", "PRODUCT_ID", "SNAPSHOT_DATE"]
}

results = []

for table_name, key_columns in tables_and_keys.items():

    df = session.table(f"NOVAMART_DB.RAW.{table_name}")

    total_rows = df.count()
    exact_unique_rows = df.drop_duplicates().count()
    unique_business_keys = df.drop_duplicates(key_columns).count()

    results.append({
        "TABLE_NAME": table_name,
        "TOTAL_ROWS": total_rows,
        "EXACT_DUPLICATES": total_rows - exact_unique_rows,
        "DUPLICATE_KEY_ROWS": total_rows - unique_business_keys
    })

duplicate_profile = session.create_dataframe(results)

duplicate_profile.show()

In [ ]:
customer_conflicts = session.sql("""
    WITH duplicate_ids AS (
        SELECT CUSTOMER_ID
        FROM NOVAMART_DB.RAW.CUSTOMERS_RAW
        GROUP BY CUSTOMER_ID
        HAVING COUNT(DISTINCT HASH(
            CUSTOMER_NAME,
            CITY,
            STATE,
            SIGNUP_DATE,
            CUSTOMER_SEGMENT
        )) > 1
    )
    SELECT c.*
    FROM NOVAMART_DB.RAW.CUSTOMERS_RAW c
    INNER JOIN duplicate_ids d
        ON c.CUSTOMER_ID = d.CUSTOMER_ID
    ORDER BY c.CUSTOMER_ID
""")

customer_conflicts.show()

In [ ]:
order_conflicts = session.sql("""
    WITH duplicate_ids AS (
        SELECT ORDER_ID
        FROM NOVAMART_DB.RAW.ORDERS_RAW
        GROUP BY ORDER_ID
        HAVING COUNT(DISTINCT HASH(
            ORDER_DATE,
            CUSTOMER_ID,
            STORE_ID,
            ORDER_STATUS,
            PAYMENT_METHOD
        )) > 1
    )
    SELECT o.*
    FROM NOVAMART_DB.RAW.ORDERS_RAW o
    INNER JOIN duplicate_ids d
        ON o.ORDER_ID = d.ORDER_ID
    ORDER BY o.ORDER_ID
""")

order_conflicts.show()

In [ ]:
order_payment_check = session.sql("""
    SELECT
        o.ORDER_ID,
        o.ORDER_DATE,
        o.ORDER_STATUS,
        o.PAYMENT_METHOD,
        p.PAYMENT_DATE,
        p.PAYMENT_STATUS,
        ABS(
            DATEDIFF(
                'SECOND',
                TRY_TO_TIMESTAMP_NTZ(o.ORDER_DATE),
                TRY_TO_TIMESTAMP_NTZ(p.PAYMENT_DATE)
            )
        ) AS TIME_DIFFERENCE_SECONDS
    FROM NOVAMART_DB.RAW.ORDERS_RAW o
    LEFT JOIN NOVAMART_DB.RAW.PAYMENTS_RAW p
        ON o.ORDER_ID = p.ORDER_ID
    WHERE o.ORDER_ID IN (
        'ORD003318',
        'ORD032961',
        'ORD063125'
    )
    ORDER BY
        o.ORDER_ID,
        TIME_DIFFERENCE_SECONDS
""")

order_payment_check.show()

In [ ]:
# Create the cleaned customer table.
# The most complete record is retained for each CUSTOMER_ID.

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.CUSTOMERS_CLEAN AS

    SELECT
        CUSTOMER_ID,
        CUSTOMER_NAME,
        CITY,
        STATE,
        SIGNUP_DATE,
        CUSTOMER_SEGMENT

    FROM NOVAMART_DB.RAW.CUSTOMERS_RAW

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY CUSTOMER_ID
        ORDER BY
            (
                IFF(CUSTOMER_NAME IS NULL, 0, 1) +
                IFF(CITY IS NULL, 0, 1) +
                IFF(STATE IS NULL, 0, 1) +
                IFF(SIGNUP_DATE IS NULL, 0, 1) +
                IFF(CUSTOMER_SEGMENT IS NULL, 0, 1)
            ) DESC
    ) = 1
""").collect()


# Create the cleaned order table.
# Status casing is standardized and the order time closest
# to its payment time is retained.

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.ORDERS_CLEAN AS

    SELECT
        o.ORDER_ID,
        o.ORDER_DATE,
        o.CUSTOMER_ID,
        o.STORE_ID,
        INITCAP(TRIM(o.ORDER_STATUS)) AS ORDER_STATUS,
        o.PAYMENT_METHOD

    FROM NOVAMART_DB.RAW.ORDERS_RAW o

    LEFT JOIN NOVAMART_DB.RAW.PAYMENTS_RAW p
        ON o.ORDER_ID = p.ORDER_ID

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY o.ORDER_ID
        ORDER BY
            ABS(
                DATEDIFF(
                    'SECOND',
                    TRY_TO_TIMESTAMP_NTZ(o.ORDER_DATE),
                    TRY_TO_TIMESTAMP_NTZ(p.PAYMENT_DATE)
                )
            ) ASC NULLS LAST
    ) = 1
""").collect()


print("CUSTOMERS_CLEAN and ORDERS_CLEAN created successfully.")

In [ ]:
session.sql("""
    SELECT
        'CUSTOMERS_CLEAN' AS TABLE_NAME,
        COUNT(*) AS ROW_COUNT,
        COUNT(DISTINCT CUSTOMER_ID) AS DISTINCT_KEYS
    FROM NOVAMART_DB.STAGING.CUSTOMERS_CLEAN

    UNION ALL

    SELECT
        'ORDERS_CLEAN',
        COUNT(*),
        COUNT(DISTINCT ORDER_ID)
    FROM NOVAMART_DB.STAGING.ORDERS_CLEAN
""").show()

In [ ]:
remaining_tables = {
    "STORES_RAW": {
        "target": "STORES_CLEAN",
        "keys": ["STORE_ID"]
    },
    "PRODUCTS_RAW": {
        "target": "PRODUCTS_CLEAN",
        "keys": ["PRODUCT_ID"]
    },
    "ORDER_DETAILS_RAW": {
        "target": "ORDER_DETAILS_CLEAN",
        "keys": ["ORDER_DETAIL_ID"]
    },
    "PAYMENTS_RAW": {
        "target": "PAYMENTS_CLEAN",
        "keys": ["PAYMENT_ID"]
    },
    "RETURNS_RAW": {
        "target": "RETURNS_CLEAN",
        "keys": ["RETURN_ID"]
    },
    "INVENTORY_RAW": {
        "target": "INVENTORY_CLEAN",
        "keys": ["STORE_ID", "PRODUCT_ID", "SNAPSHOT_DATE"]
    }
}

for source_table, config in remaining_tables.items():

    source_df = session.table(
        f"NOVAMART_DB.RAW.{source_table}"
    )

    clean_df = source_df.drop_duplicates(config["keys"])

    clean_df.write.mode("overwrite").save_as_table(
        f"NOVAMART_DB.STAGING.{config['target']}"
    )

    print(f"{config['target']} created successfully.")

In [ ]:
staging_tables = [
    "CUSTOMERS_CLEAN",
    "STORES_CLEAN",
    "PRODUCTS_CLEAN",
    "ORDERS_CLEAN",
    "ORDER_DETAILS_CLEAN",
    "PAYMENTS_CLEAN",
    "RETURNS_CLEAN",
    "INVENTORY_CLEAN"
]

results = []

for table_name in staging_tables:

    row_count = session.table(
        f"NOVAMART_DB.STAGING.{table_name}"
    ).count()

    results.append({
        "TABLE_NAME": table_name,
        "ROW_COUNT": row_count
    })

session.create_dataframe(results).show()

In [ ]:
from snowflake.snowpark.functions import (
    col,
    trim,
    when,
    sum as snowpark_sum
)

missing_results = []

for table_name in staging_tables:

    df = session.table(
        f"NOVAMART_DB.STAGING.{table_name}"
    )

    columns = df.columns

    null_expressions = [
        snowpark_sum(
            when(
                col(column_name).is_null() |
                (trim(col(column_name).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(column_name)
        for column_name in columns
    ]

    null_counts = df.select(null_expressions).collect()[0]

    for column_name in columns:

        missing_count = null_counts[column_name]

        if missing_count > 0:
            missing_results.append({
                "TABLE_NAME": table_name,
                "COLUMN_NAME": column_name,
                "MISSING_COUNT": missing_count
            })

if missing_results:
    session.create_dataframe(missing_results).show()
else:
    print("No missing values found.")

In [ ]:
quality_check = session.sql("""
    SELECT
        'ORDER_DETAILS_CLEAN' AS TABLE_NAME,
        'Invalid or zero quantity' AS ISSUE,
        COUNT_IF(
            TRY_TO_NUMBER(QUANTITY) IS NULL
            OR TRY_TO_NUMBER(QUANTITY) <= 0
        ) AS ISSUE_COUNT
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN

    UNION ALL

    SELECT
        'ORDER_DETAILS_CLEAN',
        'Invalid or negative unit price',
        COUNT_IF(
            TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) IS NULL
            OR TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) < 0
        )
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN

    UNION ALL

    SELECT
        'ORDER_DETAILS_CLEAN',
        'Discount outside 0 to 1',
        COUNT_IF(
            TRY_TO_DECIMAL(DISCOUNT, 10, 4) IS NULL
            OR TRY_TO_DECIMAL(DISCOUNT, 10, 4) < 0
            OR TRY_TO_DECIMAL(DISCOUNT, 10, 4) > 1
        )
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN

    UNION ALL

    SELECT
        'PAYMENTS_CLEAN',
        'Invalid or negative payment amount',
        COUNT_IF(
            TRY_TO_DECIMAL(AMOUNT, 14, 2) IS NULL
            OR TRY_TO_DECIMAL(AMOUNT, 14, 2) < 0
        )
    FROM NOVAMART_DB.STAGING.PAYMENTS_CLEAN

    UNION ALL

    SELECT
        'INVENTORY_CLEAN',
        'Invalid or negative stock',
        COUNT_IF(
            TRY_TO_NUMBER(STOCK_ON_HAND) IS NULL
            OR TRY_TO_NUMBER(STOCK_ON_HAND) < 0
        )
    FROM NOVAMART_DB.STAGING.INVENTORY_CLEAN

    UNION ALL

    SELECT
        'INVENTORY_CLEAN',
        'Invalid or negative reorder level',
        COUNT_IF(
            TRY_TO_NUMBER(REORDER_LEVEL) IS NULL
            OR TRY_TO_NUMBER(REORDER_LEVEL) < 0
        )
    FROM NOVAMART_DB.STAGING.INVENTORY_CLEAN

    UNION ALL

    SELECT
        'ORDERS_CLEAN',
        'Invalid order date',
        COUNT_IF(TRY_TO_TIMESTAMP_NTZ(ORDER_DATE) IS NULL)
    FROM NOVAMART_DB.STAGING.ORDERS_CLEAN

    UNION ALL

    SELECT
        'PAYMENTS_CLEAN',
        'Invalid payment date',
        COUNT_IF(TRY_TO_TIMESTAMP_NTZ(PAYMENT_DATE) IS NULL)
    FROM NOVAMART_DB.STAGING.PAYMENTS_CLEAN

    UNION ALL

    SELECT
        'RETURNS_CLEAN',
        'Invalid return date',
        COUNT_IF(TRY_TO_DATE(RETURN_DATE) IS NULL)
    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN

    UNION ALL

    SELECT
        'INVENTORY_CLEAN',
        'Invalid snapshot date',
        COUNT_IF(TRY_TO_DATE(SNAPSHOT_DATE) IS NULL)
    FROM NOVAMART_DB.STAGING.INVENTORY_CLEAN
""")

quality_check.show()

In [ ]:
invalid_values = session.sql("""
    SELECT
        'QUANTITY' AS COLUMN_NAME,
        QUANTITY AS INVALID_VALUE,
        COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN
    WHERE
        TRY_TO_NUMBER(QUANTITY) IS NULL
        OR TRY_TO_NUMBER(QUANTITY) <= 0
    GROUP BY QUANTITY

    UNION ALL

    SELECT
        'UNIT_PRICE',
        UNIT_PRICE,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN
    WHERE
        TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) IS NULL
        OR TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) < 0
    GROUP BY UNIT_PRICE

    UNION ALL

    SELECT
        'PAYMENT_AMOUNT',
        AMOUNT,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.PAYMENTS_CLEAN
    WHERE
        TRY_TO_DECIMAL(AMOUNT, 14, 2) IS NULL
        OR TRY_TO_DECIMAL(AMOUNT, 14, 2) < 0
    GROUP BY AMOUNT

    ORDER BY COLUMN_NAME, INVALID_VALUE
""")

invalid_values.show(100)

In [ ]:
session.sql("""
    SELECT
        QUANTITY AS INVALID_QUANTITY,
        COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN
    WHERE
        TRY_TO_NUMBER(QUANTITY) IS NULL
        OR TRY_TO_NUMBER(QUANTITY) <= 0
    GROUP BY QUANTITY
    ORDER BY TRY_TO_NUMBER(QUANTITY)
""").show()

In [ ]:
session.sql("""
    SELECT
        UNIT_PRICE AS INVALID_UNIT_PRICE,
        COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN
    WHERE
        TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) IS NULL
        OR TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) < 0
    GROUP BY UNIT_PRICE
    ORDER BY TRY_TO_DECIMAL(UNIT_PRICE, 12, 2)
""").show(100)

In [ ]:
# Quarantine zero-quantity order details

session.sql("""
    CREATE OR REPLACE TABLE
        NOVAMART_DB.STAGING.ORDER_DETAILS_REJECTED AS

    SELECT
        *,
        'ZERO_QUANTITY' AS REJECTION_REASON,
        CURRENT_TIMESTAMP() AS REJECTED_AT

    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN

    WHERE TRY_TO_NUMBER(QUANTITY) <= 0
       OR TRY_TO_NUMBER(QUANTITY) IS NULL
""").collect()


# Create validated order details

session.sql("""
    CREATE OR REPLACE TABLE
        NOVAMART_DB.STAGING.ORDER_DETAILS_VALIDATED AS

    SELECT
        od.ORDER_DETAIL_ID,
        od.ORDER_ID,
        od.PRODUCT_ID,
        TRY_TO_NUMBER(od.QUANTITY)::INTEGER AS QUANTITY,

        CASE
            WHEN TRY_TO_DECIMAL(od.UNIT_PRICE, 12, 2) < 0
              OR TRY_TO_DECIMAL(od.UNIT_PRICE, 12, 2) IS NULL
            THEN p.UNIT_PRICE
            ELSE TRY_TO_DECIMAL(od.UNIT_PRICE, 12, 2)
        END AS UNIT_PRICE,

        TRY_TO_DECIMAL(od.DISCOUNT, 10, 4) AS DISCOUNT,

        IFF(
            TRY_TO_DECIMAL(od.UNIT_PRICE, 12, 2) < 0
            OR TRY_TO_DECIMAL(od.UNIT_PRICE, 12, 2) IS NULL,
            TRUE,
            FALSE
        ) AS PRICE_WAS_CORRECTED

    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_CLEAN od

    LEFT JOIN NOVAMART_DB.STAGING.PRODUCTS_CLEAN p
        ON od.PRODUCT_ID = p.PRODUCT_ID

    WHERE TRY_TO_NUMBER(od.QUANTITY) > 0
""").collect()


# Create validated payments

session.sql("""
    CREATE OR REPLACE TABLE
        NOVAMART_DB.STAGING.PAYMENTS_VALIDATED AS

    SELECT
        PAYMENT_ID,
        ORDER_ID,
        TRY_TO_TIMESTAMP_NTZ(PAYMENT_DATE) AS PAYMENT_DATE,
        PAYMENT_STATUS,
        ABS(TRY_TO_DECIMAL(AMOUNT, 14, 2)) AS AMOUNT,
        IFF(
            TRY_TO_DECIMAL(AMOUNT, 14, 2) < 0,
            TRUE,
            FALSE
        ) AS AMOUNT_WAS_NEGATIVE

    FROM NOVAMART_DB.STAGING.PAYMENTS_CLEAN
""").collect()

print("Invalid numerical values processed successfully.")

In [ ]:
session.sql("""
    SELECT 'Rejected zero quantities' AS CHECK_NAME, COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_REJECTED

    UNION ALL

    SELECT 'Validated order details', COUNT(*)
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_VALIDATED

    UNION ALL

    SELECT 'Corrected unit prices', COUNT_IF(PRICE_WAS_CORRECTED)
    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_VALIDATED

    UNION ALL

    SELECT 'Validated payments', COUNT(*)
    FROM NOVAMART_DB.STAGING.PAYMENTS_VALIDATED

    UNION ALL

    SELECT 'Corrected negative payments', COUNT_IF(AMOUNT_WAS_NEGATIVE)
    FROM NOVAMART_DB.STAGING.PAYMENTS_VALIDATED
""").show()

In [ ]:
category_profile = session.sql("""
    SELECT
        'CUSTOMER_SEGMENT' AS COLUMN_NAME,
        CUSTOMER_SEGMENT AS COLUMN_VALUE,
        COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.CUSTOMERS_CLEAN
    GROUP BY CUSTOMER_SEGMENT

    UNION ALL

    SELECT
        'PRODUCT_STATUS',
        PRODUCT_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.PRODUCTS_CLEAN
    GROUP BY PRODUCT_STATUS

    UNION ALL

    SELECT
        'ORDER_STATUS',
        ORDER_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.ORDERS_CLEAN
    GROUP BY ORDER_STATUS

    UNION ALL

    SELECT
        'PAYMENT_METHOD',
        PAYMENT_METHOD,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.ORDERS_CLEAN
    GROUP BY PAYMENT_METHOD

    UNION ALL

    SELECT
        'PAYMENT_STATUS',
        PAYMENT_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.PAYMENTS_VALIDATED
    GROUP BY PAYMENT_STATUS

    UNION ALL

    SELECT
        'REFUND_STATUS',
        REFUND_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN
    GROUP BY REFUND_STATUS

    UNION ALL

    SELECT
        'RETURN_REASON',
        RETURN_REASON,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN
    GROUP BY RETURN_REASON

    ORDER BY COLUMN_NAME, COLUMN_VALUE
""")

category_profile.show(100)

In [ ]:
session.sql("""
    SELECT
        'PRODUCT_STATUS' AS COLUMN_NAME,
        PRODUCT_STATUS AS COLUMN_VALUE,
        COUNT(*) AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.PRODUCTS_CLEAN
    GROUP BY PRODUCT_STATUS

    UNION ALL

    SELECT
        'PAYMENT_STATUS',
        PAYMENT_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.PAYMENTS_VALIDATED
    GROUP BY PAYMENT_STATUS

    UNION ALL

    SELECT
        'REFUND_STATUS',
        REFUND_STATUS,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN
    GROUP BY REFUND_STATUS

    UNION ALL

    SELECT
        'RETURN_REASON',
        RETURN_REASON,
        COUNT(*)
    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN
    GROUP BY RETURN_REASON

    ORDER BY COLUMN_NAME, COLUMN_VALUE
""").show(50)

In [ ]:
# Final standardized customer table

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_CUSTOMERS AS

    WITH city_state_counts AS (
        SELECT
            CITY,
            STATE,
            COUNT(*) AS RECORD_COUNT
        FROM NOVAMART_DB.STAGING.CUSTOMERS_CLEAN
        WHERE CITY IS NOT NULL
          AND TRIM(CITY) <> ''
          AND STATE IS NOT NULL
          AND TRIM(STATE) <> ''
        GROUP BY CITY, STATE
    ),

    city_state_map AS (
        SELECT CITY, STATE
        FROM city_state_counts
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY CITY
            ORDER BY RECORD_COUNT DESC
        ) = 1
    )

    SELECT
        c.CUSTOMER_ID,
        TRIM(c.CUSTOMER_NAME) AS CUSTOMER_NAME,
        COALESCE(NULLIF(TRIM(c.CITY), ''), 'Unknown') AS CITY,

        COALESCE(
            NULLIF(TRIM(c.STATE), ''),
            m.STATE,
            'Unknown'
        ) AS STATE,

        TRY_TO_DATE(c.SIGNUP_DATE) AS SIGNUP_DATE,
        INITCAP(TRIM(c.CUSTOMER_SEGMENT)) AS CUSTOMER_SEGMENT,

        IFF(
            c.CITY IS NULL OR TRIM(c.CITY) = '',
            TRUE,
            FALSE
        ) AS CITY_WAS_MISSING,

        IFF(
            c.STATE IS NULL OR TRIM(c.STATE) = '',
            TRUE,
            FALSE
        ) AS STATE_WAS_MISSING

    FROM NOVAMART_DB.STAGING.CUSTOMERS_CLEAN c

    LEFT JOIN city_state_map m
        ON c.CITY = m.CITY
""").collect()


# Final standardized product table

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_PRODUCTS AS

    SELECT
        PRODUCT_ID,

        COALESCE(
            NULLIF(TRIM(PRODUCT_NAME), ''),
            'Unknown Product - ' || PRODUCT_ID
        ) AS PRODUCT_NAME,

        INITCAP(TRIM(CATEGORY)) AS CATEGORY,
        TRY_TO_DECIMAL(UNIT_PRICE, 12, 2) AS UNIT_PRICE,
        INITCAP(TRIM(PRODUCT_STATUS)) AS PRODUCT_STATUS,

        IFF(
            PRODUCT_NAME IS NULL OR TRIM(PRODUCT_NAME) = '',
            TRUE,
            FALSE
        ) AS PRODUCT_NAME_WAS_MISSING

    FROM NOVAMART_DB.STAGING.PRODUCTS_CLEAN
""").collect()


# Final standardized orders table

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_ORDERS AS

    SELECT
        ORDER_ID,
        TRY_TO_TIMESTAMP_NTZ(ORDER_DATE) AS ORDER_DATE,

        COALESCE(
            NULLIF(TRIM(CUSTOMER_ID), ''),
            'UNKNOWN_CUSTOMER'
        ) AS CUSTOMER_ID,

        COALESCE(
            NULLIF(TRIM(STORE_ID), ''),
            'UNKNOWN_STORE'
        ) AS STORE_ID,

        INITCAP(TRIM(ORDER_STATUS)) AS ORDER_STATUS,

        CASE UPPER(TRIM(PAYMENT_METHOD))
            WHEN 'UPI' THEN 'UPI'
            WHEN 'CASH' THEN 'Cash'
            WHEN 'CREDIT CARD' THEN 'Credit Card'
            WHEN 'DEBIT CARD' THEN 'Debit Card'
            ELSE INITCAP(TRIM(PAYMENT_METHOD))
        END AS PAYMENT_METHOD,

        IFF(
            CUSTOMER_ID IS NULL OR TRIM(CUSTOMER_ID) = '',
            TRUE,
            FALSE
        ) AS CUSTOMER_ID_WAS_MISSING,

        IFF(
            STORE_ID IS NULL OR TRIM(STORE_ID) = '',
            TRUE,
            FALSE
        ) AS STORE_ID_WAS_MISSING

    FROM NOVAMART_DB.STAGING.ORDERS_CLEAN
""").collect()

print("Customer, product and order staging tables created.")

In [ ]:
session.sql("""
    SELECT
        'Customer cities labelled Unknown' AS CHECK_NAME,
        COUNT_IF(CITY = 'Unknown') AS RECORD_COUNT
    FROM NOVAMART_DB.STAGING.STG_CUSTOMERS

    UNION ALL

    SELECT
        'Customer states still Unknown',
        COUNT_IF(STATE = 'Unknown')
    FROM NOVAMART_DB.STAGING.STG_CUSTOMERS

    UNION ALL

    SELECT
        'Unknown product names',
        COUNT_IF(PRODUCT_NAME_WAS_MISSING)
    FROM NOVAMART_DB.STAGING.STG_PRODUCTS

    UNION ALL

    SELECT
        'Unknown customer IDs',
        COUNT_IF(CUSTOMER_ID_WAS_MISSING)
    FROM NOVAMART_DB.STAGING.STG_ORDERS

    UNION ALL

    SELECT
        'Unknown store IDs',
        COUNT_IF(STORE_ID_WAS_MISSING)
    FROM NOVAMART_DB.STAGING.STG_ORDERS
""").show()

In [ ]:
session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_STORES AS

    SELECT
        STORE_ID,
        TRIM(STORE_NAME) AS STORE_NAME,
        INITCAP(TRIM(CITY)) AS CITY,
        INITCAP(TRIM(STATE)) AS STATE,
        INITCAP(TRIM(REGION)) AS STORE_TYPE

    FROM NOVAMART_DB.STAGING.STORES_CLEAN
""").collect()

In [ ]:
# Order details

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_ORDER_DETAILS AS

    SELECT
        ORDER_DETAIL_ID,
        ORDER_ID,
        PRODUCT_ID,
        QUANTITY,
        UNIT_PRICE,
        DISCOUNT,
        ROUND(QUANTITY * UNIT_PRICE, 2) AS GROSS_AMOUNT,
        ROUND(QUANTITY * UNIT_PRICE * DISCOUNT, 2) AS DISCOUNT_AMOUNT,
        ROUND(QUANTITY * UNIT_PRICE * (1 - DISCOUNT), 2) AS NET_AMOUNT,
        PRICE_WAS_CORRECTED

    FROM NOVAMART_DB.STAGING.ORDER_DETAILS_VALIDATED
""").collect()


# Payments

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_PAYMENTS AS

    SELECT
        PAYMENT_ID,
        ORDER_ID,
        PAYMENT_DATE,
        INITCAP(TRIM(PAYMENT_STATUS)) AS PAYMENT_STATUS,
        AMOUNT,
        AMOUNT_WAS_NEGATIVE

    FROM NOVAMART_DB.STAGING.PAYMENTS_VALIDATED
""").collect()


# Returns

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_RETURNS AS

    SELECT
        RETURN_ID,
        ORDER_ID,
        TRY_TO_DATE(RETURN_DATE) AS RETURN_DATE,
        INITCAP(TRIM(RETURN_REASON)) AS RETURN_REASON,
        INITCAP(TRIM(REFUND_STATUS)) AS REFUND_STATUS

    FROM NOVAMART_DB.STAGING.RETURNS_CLEAN
""").collect()


# Inventory

session.sql("""
    CREATE OR REPLACE TABLE NOVAMART_DB.STAGING.STG_INVENTORY AS

    SELECT
        STORE_ID,
        PRODUCT_ID,
        TRY_TO_NUMBER(STOCK_ON_HAND)::INTEGER AS STOCK_ON_HAND,
        TRY_TO_NUMBER(REORDER_LEVEL)::INTEGER AS REORDER_LEVEL,
        TRY_TO_DATE(SNAPSHOT_DATE) AS SNAPSHOT_DATE

    FROM NOVAMART_DB.STAGING.INVENTORY_CLEAN
""").collect()

print("Remaining staging tables created successfully.")

In [ ]:
final_staging_tables = [
    "STG_CUSTOMERS",
    "STG_STORES",
    "STG_PRODUCTS",
    "STG_ORDERS",
    "STG_ORDER_DETAILS",
    "STG_PAYMENTS",
    "STG_RETURNS",
    "STG_INVENTORY"
]

final_counts = []

for table_name in final_staging_tables:

    row_count = session.table(
        f"NOVAMART_DB.STAGING.{table_name}"
    ).count()

    final_counts.append({
        "TABLE_NAME": table_name,
        "ROW_COUNT": row_count
    })

session.create_dataframe(final_counts).show()

In [ ]:
referential_checks = session.sql("""
    SELECT
        'Orders with nonexistent customers' AS CHECK_NAME,
        COUNT(*) AS ISSUE_COUNT
    FROM NOVAMART_DB.STAGING.STG_ORDERS o
    LEFT JOIN NOVAMART_DB.STAGING.STG_CUSTOMERS c
        ON o.CUSTOMER_ID = c.CUSTOMER_ID
    WHERE c.CUSTOMER_ID IS NULL
      AND o.CUSTOMER_ID <> 'UNKNOWN_CUSTOMER'

    UNION ALL

    SELECT
        'Orders with nonexistent stores',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_ORDERS o
    LEFT JOIN NOVAMART_DB.STAGING.STG_STORES s
        ON o.STORE_ID = s.STORE_ID
    WHERE s.STORE_ID IS NULL
      AND o.STORE_ID <> 'UNKNOWN_STORE'

    UNION ALL

    SELECT
        'Order details with nonexistent orders',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_ORDER_DETAILS d
    LEFT JOIN NOVAMART_DB.STAGING.STG_ORDERS o
        ON d.ORDER_ID = o.ORDER_ID
    WHERE o.ORDER_ID IS NULL

    UNION ALL

    SELECT
        'Order details with nonexistent products',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_ORDER_DETAILS d
    LEFT JOIN NOVAMART_DB.STAGING.STG_PRODUCTS p
        ON d.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.PRODUCT_ID IS NULL

    UNION ALL

    SELECT
        'Payments with nonexistent orders',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_PAYMENTS p
    LEFT JOIN NOVAMART_DB.STAGING.STG_ORDERS o
        ON p.ORDER_ID = o.ORDER_ID
    WHERE o.ORDER_ID IS NULL

    UNION ALL

    SELECT
        'Returns with nonexistent orders',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_RETURNS r
    LEFT JOIN NOVAMART_DB.STAGING.STG_ORDERS o
        ON r.ORDER_ID = o.ORDER_ID
    WHERE o.ORDER_ID IS NULL

    UNION ALL

    SELECT
        'Inventory with nonexistent stores',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_INVENTORY i
    LEFT JOIN NOVAMART_DB.STAGING.STG_STORES s
        ON i.STORE_ID = s.STORE_ID
    WHERE s.STORE_ID IS NULL

    UNION ALL

    SELECT
        'Inventory with nonexistent products',
        COUNT(*)
    FROM NOVAMART_DB.STAGING.STG_INVENTORY i
    LEFT JOIN NOVAMART_DB.STAGING.STG_PRODUCTS p
        ON i.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.PRODUCT_ID IS NULL
""")

referential_checks.show()